# Phase 2 Loss-Component Analysis

## Experiment Design

### Phase 1: Loss-Component Logging (必做)
- **Purpose**: Verify real loss term magnitudes, decide if normalization is needed
- **Runs**: 3 losses × 1 seed = 3 runs (~3 hours)
- **Losses**: m2_robust_gamma07, m2_robust_gamma10, imadl_m2_alpha06
- **Seed**: 42

### Phase 2: Normalization Experiment (条件执行)
- **Condition**: Only if scale_ratio > 10 from Phase 1
- **Runs**: 3 normalized losses × 3 seeds = 9 runs (~9 hours)
- **Method**: Batch normalization of loss components

## Decision Rules

**Phase 1**:
- `scale_ratio < 10`: Loss-scale balanced, skip Phase 2
- `10 ≤ scale_ratio < 30`: Moderate imbalance, normalization optional
- `scale_ratio ≥ 30`: Severe imbalance, normalization required

**Phase 2**:
- If `normalized_sharpe > original_sharpe`: Use normalized version
- Else: Keep original version

## Setup: Mount Drive and Clone Repo

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify Drive mount
!ls -la /content/drive/MyDrive/FYP

In [ ]:
%%bash
set -euo pipefail

# Clone or update FYP repo
if [ -d "/content/FYP" ]; then
    cd /content/FYP
    git fetch origin
    git checkout phase2.2-fix
    git pull origin phase2.2-fix
else
    cd /content
    git clone https://github.com/ROUCHER27/FYP.git FYP
    cd FYP
    git checkout phase2.2-fix
fi

# Print branch and commit
echo "Branch: $(git branch --show-current)"
echo "Commit: $(git rev-parse --short HEAD)"

## Install Dependencies

In [ ]:
%%bash
cd /content/FYP
pip install -q -r requirements.txt

## Configuration

In [ ]:
# Experiment configuration
BRANCH = "phase2.2-fix"
DRIVE_ROOT = "/content/drive/MyDrive/FYP"
OUTPUT_ROOT = f"{DRIVE_ROOT}/{BRANCH}"
LOG_ROOT = f"{OUTPUT_ROOT}/logs"

# Phase 1 configuration
PHASE1_LOSSES = [
    "m2_robust_gamma07",
    "m2_robust_gamma10",
    "imadl_m2_alpha06"
]
PHASE1_SEED = 42

# Phase 2 configuration (conditional)
PHASE2_SEEDS = [42, 123, 456]

# Phase 2.2 diagnostics estimates (from existing results)
# These are used as estimates since loss-component logging is not yet implemented
PHASE22_SCALE_RATIOS = {
    "m2_robust_gamma07": 113.0,  # From Phase 2.2 diagnostics
    "m2_robust_gamma10": 113.0,  # From Phase 2.2 diagnostics
    "imadl_m2_alpha06": 34.0     # From Phase 2.2 diagnostics
}

# Create output directories
!mkdir -p {OUTPUT_ROOT}/results
!mkdir -p {LOG_ROOT}

print(f"Output root: {OUTPUT_ROOT}")
print(f"Log root: {LOG_ROOT}")

## Phase 1: Loss-Component Logging

Run 3 losses with component logging to measure scale ratios.

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP

# Hardcoded paths (matching Cell 7 configuration)
OUTPUT_ROOT="/content/drive/MyDrive/FYP/phase2.2-fix"
LOG_ROOT="/content/drive/MyDrive/FYP/phase2.2-fix/logs"

# Ensure LOG_ROOT directory exists
mkdir -p "$LOG_ROOT"

# Phase 1 losses
LOSSES=("m2_robust_gamma07" "m2_robust_gamma10" "imadl_m2_alpha06")
SEED=42

for loss in "${LOSSES[@]}"; do
    echo "========================================"
    echo "Running Phase 1: $loss (seed=$SEED)"
    echo "========================================"
    
    RUN_ID="${loss}_seed${SEED}"
    OUTPUT_DIR="$OUTPUT_ROOT/results/$RUN_ID"
    LOG_FILE="$LOG_ROOT/${RUN_ID}.log"
    
    # Check if already completed
    if [ -f "$OUTPUT_DIR/sanity_metrics_${loss}.csv" ]; then
        echo "✓ Already completed: $RUN_ID"
        continue
    fi
    
    # Run training with all required parameters
    # NOTE: Loss-component logging not yet implemented, using Phase 2.2 diagnostics estimates instead
    python sanity_check_signal_tilted.py \
        --loss "$loss" \
        --seed "$SEED" \
        --output-dir "$OUTPUT_DIR" \
        --data-dir /content/FYP \
        --train-start "1990-01" \
        --train-end "1994-12" \
        --test-start "1995-01" \
        --test-months 24 \
        --max-epochs 20 \
        --batch-size 1024 \
        2>&1 | tee "$LOG_FILE"
    
    if [ $? -eq 0 ]; then
        echo "✓ Completed: $RUN_ID"
    else
        echo "✗ Failed: $RUN_ID"
    fi
done

echo "========================================"
echo "Phase 1 Complete"
echo "======================================="

## Analyze Phase 1 Results

In [ ]:
import pandas as pd
import json
from pathlib import Path

# NOTE: Loss-component logging not yet implemented in sanity_check_signal_tilted.py
# Using Phase 2.2 diagnostics estimates instead (from docs/phase2/phase2.2_diagnostics.md)

print("Using Phase 2.2 diagnostics estimates for scale_ratio:")
print("=" * 50)

phase1_results = {}

for loss in PHASE1_LOSSES:
    # Use pre-calculated scale ratios from Phase 2.2 diagnostics
    scale_ratio = PHASE22_SCALE_RATIOS[loss]
    
    phase1_results[loss] = {
        'scale_ratio': scale_ratio,
        'source': 'Phase 2.2 diagnostics estimate'
    }
    
    print(f"\n{loss}:")
    print(f"  scale_ratio: {scale_ratio:.2f} (from Phase 2.2 diagnostics)")

# Calculate average scale ratio
avg_scale_ratio = sum(r['scale_ratio'] for r in phase1_results.values()) / len(phase1_results)
print(f"\n{'='*50}")
print(f"Average scale_ratio: {avg_scale_ratio:.2f}")
print(f"{'='*50}")

# Save Phase 1 summary
summary_path = Path(OUTPUT_ROOT) / "phase1_summary.json"
with open(summary_path, 'w') as f:
    json.dump({
        'results': phase1_results,
        'avg_scale_ratio': avg_scale_ratio,
        'note': 'Scale ratios estimated from Phase 2.2 diagnostics (loss-component logging not yet implemented)'
    }, f, indent=2)

print(f"\nPhase 1 summary saved to: {summary_path}")
print("\nNOTE: These are estimates from Phase 2.2 diagnostics.")
print("TODO: Implement loss-component logging in sanity_check_signal_tilted.py for exact measurements.")

## Decision: Should We Run Phase 2?

In [ ]:
# Decision logic
if avg_scale_ratio < 10:
    decision = "SKIP_PHASE2"
    reason = "Loss-scale balanced, no normalization needed"
    print(f"\n✓ {reason}")
    print(f"  scale_ratio = {avg_scale_ratio:.2f} < 10")
    print(f"\n→ Experiment complete. Use original losses.")
elif avg_scale_ratio < 30:
    decision = "OPTIONAL_PHASE2"
    reason = "Moderate imbalance, normalization optional"
    print(f"\n⚠ {reason}")
    print(f"  10 ≤ scale_ratio = {avg_scale_ratio:.2f} < 30")
    print(f"\n→ You may run Phase 2 to test normalization, but it's not required.")
else:
    decision = "RUN_PHASE2"
    reason = "Severe imbalance, normalization required"
    print(f"\n✗ {reason}")
    print(f"  scale_ratio = {avg_scale_ratio:.2f} ≥ 30")
    print(f"\n→ Proceeding to Phase 2...")

# Save decision
decision_path = Path(OUTPUT_ROOT) / "phase1_decision.json"
with open(decision_path, 'w') as f:
    json.dump({
        'decision': decision,
        'reason': reason,
        'avg_scale_ratio': avg_scale_ratio
    }, f, indent=2)

RUN_PHASE2 = (decision == "RUN_PHASE2")
print(f"\nRUN_PHASE2 = {RUN_PHASE2}")

## Phase 2: Normalization Experiment (Conditional)

Only run if `RUN_PHASE2 == True`.

In [ ]:
if not RUN_PHASE2:
    print("Skipping Phase 2 (not required based on Phase 1 results)")
else:
    print("Starting Phase 2: Normalization Experiment")
    print(f"Running 3 losses × 3 seeds = 9 runs (~9 hours)")

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP

# Hardcoded paths (matching Cell 7 configuration)
OUTPUT_ROOT="/content/drive/MyDrive/FYP/phase2.2-fix"
LOG_ROOT="/content/drive/MyDrive/FYP/phase2.2-fix/logs"

# Ensure directories exist
mkdir -p "$LOG_ROOT"

# Phase 2: Normalized loss variants
LOSSES=("m2_robust_gamma07_normalized" "m2_robust_gamma10_normalized" "imadl_m2_alpha06_normalized")
SEEDS=(42 123 456)

for loss in "${LOSSES[@]}"; do
    for seed in "${SEEDS[@]}"; do
        echo "========================================"
        echo "Running Phase 2: $loss (seed=$seed)"
        echo "========================================"
        
        RUN_ID="${loss}_seed${seed}"
        OUTPUT_DIR="$OUTPUT_ROOT/results/$RUN_ID"
        LOG_FILE="$LOG_ROOT/${RUN_ID}.log"
        
        # Check if already completed
        if [ -f "$OUTPUT_DIR/sanity_metrics_${loss}.csv" ]; then
            echo "✓ Already completed: $RUN_ID"
            continue
        fi
        
        python sanity_check_signal_tilted.py \
            --loss "$loss" \
            --seed "$seed" \
            --output-dir "$OUTPUT_DIR" \
            --data-dir /content/FYP \
            --train-start "1990-01" \
            --train-end "1994-12" \
            --test-start "1995-01" \
            --test-months 24 \
            --max-epochs 20 \
            --batch-size 1024 \
            2>&1 | tee "$LOG_FILE"
        
        if [ $? -eq 0 ]; then
            echo "✓ Completed: $RUN_ID"
        else
            echo "✗ Failed: $RUN_ID"
        fi
    done
done

echo "========================================"
echo "Phase 2 Complete (3 losses × 3 seeds = 9 runs)"
echo "========================================"

## Analyze Phase 2 Results

In [ ]:
import pandas as pd
from pathlib import Path

def load_phase22_sharpe(loss_name):
    """Load Sharpe ratio from Phase 2.2 robustness test results.

    Original results come from two different notebooks on Drive:
    - m2_robust_gamma10, imadl_m2_alpha06: Phase2_Fixes_Colab_Runner (phase2-fixes/results)
    - m2_robust_gamma07: Phase2_2 gamma_refinement (phase2_2/gamma_refinement/results)

    All used seeds 42, 52, 62 with cap 0.05.
    Returns average Sharpe across the 3 seeds.
    """
    drive = Path("/content/drive/MyDrive/FYP")
    seeds = [42, 52, 62]

    # Map each loss to its results directory on Drive
    result_dirs = {
        "m2_robust_gamma07": drive / "phase2_2" / "gamma_refinement" / "results",
        "m2_robust_gamma10": drive / "phase2-fixes" / "results",
        "imadl_m2_alpha06":  drive / "phase2-fixes" / "results",
    }

    phase22_root = result_dirs.get(loss_name)
    if phase22_root is None:
        print(f"Error: Unknown loss {loss_name}, no results directory mapped")
        return None

    sharpes = []
    for seed in seeds:
        run_id = f"{loss_name}_seed{seed}_cap05"
        metrics_file = phase22_root / run_id / f"sanity_metrics_{loss_name}.csv"

        if metrics_file.exists():
            df = pd.read_csv(metrics_file)
            sharpe = df['sharpe'].iloc[-1]
            sharpes.append(sharpe)
        else:
            print(f"Warning: not found: {metrics_file}")

    if sharpes:
        avg_sharpe = sum(sharpes) / len(sharpes)
        print(f"Loaded Phase 2.2 Sharpe for {loss_name}: {avg_sharpe:.4f} (from {len(sharpes)} seeds)")
        return avg_sharpe
    else:
        print(f"Error: No Phase 2.2 results found for {loss_name}")
        return None


if not RUN_PHASE2:
    print("No Phase 2 results to analyze")
else:
    # Load Phase 2 results
    phase2_results = {}

    for loss in PHASE1_LOSSES:
        normalized_loss = f"{loss}_normalized"

        # Collect normalized results (from current notebook Phase 2)
        normalized_sharpes = []
        for seed in PHASE2_SEEDS:
            run_id = f"{normalized_loss}_seed{seed}"
            metrics_file = Path(OUTPUT_ROOT) / "results" / run_id / f"sanity_metrics_{normalized_loss}.csv"

            if metrics_file.exists():
                df = pd.read_csv(metrics_file)
                sharpe = df['sharpe'].iloc[-1]
                normalized_sharpes.append(sharpe)

        # Load original result from Phase 2.2 robustness test
        original_sharpe = load_phase22_sharpe(loss)

        if normalized_sharpes:
            avg_normalized_sharpe = sum(normalized_sharpes) / len(normalized_sharpes)
            phase2_results[loss] = {
                'original_sharpe': original_sharpe,
                'normalized_sharpes': normalized_sharpes,
                'avg_normalized_sharpe': avg_normalized_sharpe
            }

            print(f"\n{loss}:")
            print(f"  Original Sharpe: {original_sharpe}")
            print(f"  Normalized Sharpe (avg): {avg_normalized_sharpe:.4f}")
            print(f"  Normalized Sharpes: {normalized_sharpes}")

    # Save Phase 2 summary
    summary_path = Path(OUTPUT_ROOT) / "phase2_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(phase2_results, f, indent=2)

    print(f"\nPhase 2 summary saved to: {summary_path}")

## Final Recommendation

In [ ]:
if not RUN_PHASE2:
    print("\n" + "="*50)
    print("FINAL RECOMMENDATION")
    print("="*50)
    print(f"\nLoss-scale is balanced (scale_ratio = {avg_scale_ratio:.2f} < 10)")
    print("\n→ Use original losses without normalization")
    print("\nRecommended model: m2_robust_gamma07")
else:
    print("\n" + "="*50)
    print("FINAL RECOMMENDATION")
    print("="*50)

    wins = 0
    total = 0

    for loss, data in phase2_results.items():
        orig = data.get('original_sharpe')
        norm = data.get('avg_normalized_sharpe')

        if orig is None or norm is None:
            print(f"\n{loss}: SKIP (missing data)")
            continue

        total += 1
        delta = norm - orig
        pct = (delta / abs(orig) * 100) if orig != 0 else 0
        winner = "NORMALIZED" if norm > orig else "ORIGINAL"

        if norm > orig:
            wins += 1

        print(f"\n{loss}:")
        print(f"  Original Sharpe:   {orig:.4f}")
        print(f"  Normalized Sharpe: {norm:.4f}")
        print(f"  Delta: {delta:+.4f} ({pct:+.1f}%)")
        print(f"  → Winner: {winner}")

    print("\n" + "="*50)
    if total == 0:
        print("No valid comparisons (missing data)")
    elif wins > total / 2:
        print(f"Normalized wins {wins}/{total} → Use normalized losses")
    else:
        print(f"Original wins {total - wins}/{total} → Keep original losses")
    print("="*50)